**Load data**

In [2]:
import pandas as pd

df = pd.read_csv("/content/clean_data_after_eda.csv")

In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14606 entries, 0 to 14605
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              14606 non-null  object 
 1   channel_sales                   14606 non-null  object 
 2   cons_12m                        14606 non-null  int64  
 3   cons_gas_12m                    14606 non-null  int64  
 4   cons_last_month                 14606 non-null  int64  
 5   date_activ                      14606 non-null  object 
 6   date_end                        14606 non-null  object 
 7   date_modif_prod                 14606 non-null  object 
 8   date_renewal                    14606 non-null  object 
 9   forecast_cons_12m               14606 non-null  float64
 10  forecast_cons_year              14606 non-null  int64  
 11  forecast_discount_energy        14606 non-null  float64
 12  forecast_meter_rent_12m         

In [4]:
print(df.head())

                                 id                     channel_sales  \
0  24011ae4ebbe3035111d65fa7c15bc57  foosdfpfkusacimwkcsosbicdxkicaua   
1  d29c2c54acc38ff3c0614d0a653813dd                           MISSING   
2  764c75f661154dac3a6c254cd082ea7d  foosdfpfkusacimwkcsosbicdxkicaua   
3  bba03439a292a1e166f80264c16191cb  lmkebamcaaclubfxadlmueccxoimlema   
4  149d57cf92fc41cf94415803a877cb4b                           MISSING   

   cons_12m  cons_gas_12m  cons_last_month  date_activ    date_end  \
0         0         54946                0  2013-06-15  2016-06-15   
1      4660             0                0  2009-08-21  2016-08-30   
2       544             0                0  2010-04-16  2016-04-16   
3      1584             0                0  2010-03-30  2016-03-30   
4      4425             0              526  2010-01-13  2016-03-07   

  date_modif_prod date_renewal  forecast_cons_12m  ...  \
0      2015-11-01   2015-06-23               0.00  ...   
1      2009-08-21   2015

**Convert date columns to datetime**

In [5]:
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

**Create Estelle's feature - price difference**

In [6]:
# Assuming columns 'price_dec' and 'price_jan' exist (merged from price data)
if 'price_dec' in df.columns and 'price_jan' in df.columns:
    df['dec_jan_price_diff'] = df['price_dec'] - df['price_jan']
else:
    print("⚠️ Columns for December and January prices not found. Make sure to merge them first if needed.")

⚠️ Columns for December and January prices not found. Make sure to merge them first if needed.


**Create additional new features**

**Contract tenure in days**

In [7]:
df['contract_tenure_days'] = (df['date_end'] - df['date_activ']).dt.days

**Days since last modification**

In [8]:
today = pd.Timestamp.today()
df['days_since_modif'] = (today - df['date_modif_prod']).dt.days

**Days to next renewal**

In [9]:
df['days_to_renewal'] = (df['date_renewal'] - today).dt.days

**Consumption ratio (last month vs average monthly)**

In [10]:
df['cons_ratio'] = df['cons_last_month'] / (df['cons_12m'] / 12 + 1e-6)

**Forecast error**

In [11]:
df['forecast_error'] = df['forecast_cons_12m'] - df['cons_12m']

**Margin per kWh**

In [12]:
df['margin_per_unit'] = df['net_margin'] / (df['cons_12m'] + 1e-6)

**Price spread (peak - off-peak forecast)**

In [13]:
if 'forecast_price_energy_peak' in df.columns and 'forecast_price_energy_off_peak' in df.columns:
    df['price_spread'] = df['forecast_price_energy_peak'] - df['forecast_price_energy_off_peak']

**Gas service flag**

In [19]:
df['has_gas_flag'] = df['has_gas'].map({'t': 1, 'f': 0})

**Check new features**

In [22]:
print(df[['contract_tenure_days', 'days_since_modif', 'days_to_renewal',
          'cons_ratio', 'forecast_error', 'margin_per_unit',
          'price_spread', 'has_gas_flag']].head())

   contract_tenure_days  days_since_modif  days_to_renewal  cons_ratio  \
0                  1096              3539            -3671    0.000000   
1                  2566              5802            -3602    0.000000   
2                  2192              5564            -3738    0.000000   
3                  2192              5581            -3755    0.000000   
4                  2245              5657            -3777    1.426441   

   forecast_error  margin_per_unit  price_spread  has_gas_flag  
0            0.00     6.789900e+08     -0.016339             1  
1        -4470.05     4.053648e-03     -0.145711             0  
2         -496.04     1.213235e-02     -0.077895             0  
3        -1343.96     1.607323e-02     -0.146694             0  
4        -3979.25     1.084294e-02     -0.016885             0  


**Remove unnecessary columns (optional)**

In [23]:
#remove any columns that have only one unique value
for col in df.columns:
    if df[col].nunique() == 1:
        df.drop(columns=col, inplace=True)
        print(f"Column '{col}' dropped (only one unique value)")

**Save the final dataset**

In [24]:
df.to_csv("clean_data_with_features.csv", index=False)
print("✅ New data with engineered features saved as 'clean_data_with_features.csv'")

✅ New data with engineered features saved as 'clean_data_with_features.csv'
